# Body Fat: exact enumeration

Enumerates all $2^{13}=8192$ models under the collapsed posterior ($m_0=3$, $\tau^2=1$) and writes the exact PIPs and model probabilities to `bodyfat_exact_primal_by_expected_size`. Run this before post-processing.

## Settings

In [ ]:
from pathlib import Path
import re
import time
import gc
import numpy as np
import pandas as pd
from scipy.special import gammaln, betaln, logsumexp
DATA_PATH = Path('bodyfat.csv')
OUT_DIR = Path('bodyfat_exact_primal_by_expected_size')
TAU2 = 1.0
PRIOR_PREC = 1.0 / TAU2
EXPECTED_MODEL_SIZE_GRID = [3.0]
B_PI_FIXED = 1.0
STANDARDISE_X = True
PIP_THRESHOLD = 0.5
TOP_MODELS_TO_SAVE = 20
RUNTIME_REPEATS = 5
OUT_DIR.mkdir(exist_ok=True)
for old_file in OUT_DIR.glob('exact_model_posterior_*.csv'):
    old_file.unlink()
PREDICTORS = [
    'age',
    'weight',
    'height',
    'neck',
    'chest',
    'abdomen',
    'hip',
    'thigh',
    'knee',
    'ankle',
    'biceps',
    'forearm',
    'wrist',
]
RESPONSE_CANDIDATES = [
    'brozek_fat',
    'body_fat',
    'bodyfat',
    'brozek',
    'percent_body_fat',
    'pct_bfat',
    'pctbfat',
    'fat',
]


## Read the Body Fat data

In [ ]:
def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace(".", "_", regex=False)
        .str.replace("-", "_", regex=False)
        .str.replace("(", "", regex=False)
        .str.replace(")", "", regex=False)
    )
    return df

def read_bodyfat(path: Path) -> pd.DataFrame:
    """
    Reads one of:
      1. clean model-only CSV: response + 13 predictors;
      2. clean CSV with case_id + response + 13 predictors;
      3. raw Johnson/JSE-style CSV with extra columns.

    Returns body_fat + 13 predictors.
    """
    if not path.exists():
        raise FileNotFoundError(f'Cannot find {path}. Put this script in the same folder as bodyfat.csv, or change DATA_PATH at the top of the script.')
    df = pd.read_csv(path)
    df = clean_columns(df)
    if 'case_id' in df.columns:
        df = df.loc[df['case_id'] != 42].copy()
    elif len(df) == 252:
        df = df.drop(df.index[41]).copy()
    df = df.reset_index(drop=True)
    response_col = None
    for cand in RESPONSE_CANDIDATES:
        if cand in df.columns:
            response_col = cand
            break
    if response_col is not None and all((x in df.columns for x in PREDICTORS)):
        out = df[[response_col] + PREDICTORS].copy()
        return out.rename(columns={response_col: 'body_fat'})
    if df.shape[1] == 14:
        out = df.copy()
        out.columns = ['body_fat'] + PREDICTORS
        return out
    if df.shape[1] == 15:
        out = df.iloc[:, 1:].copy()
        out.columns = ['body_fat'] + PREDICTORS
        return out
    raw = df.copy()
    if raw.shape[1] > 18:
        first = pd.to_numeric(raw.iloc[:, 0], errors='coerce').to_numpy()
        if np.all(np.isfinite(first)) and len(np.unique(first)) == len(first):
            raw = raw.iloc[:, 1:].copy()
    if raw.shape[1] >= 18:
        raw = raw.iloc[:, :18].copy()
        raw.columns = [
            'density',
            'brozek_fat',
            'siri_fat',
            'age',
            'weight',
            'height',
            'adiposity',
            'fat_free_weight',
            'neck',
            'chest',
            'abdomen',
            'hip',
            'thigh',
            'knee',
            'ankle',
            'biceps',
            'forearm',
            'wrist',
        ]
        out = raw[['brozek_fat'] + PREDICTORS].copy()
        return out.rename(columns={'brozek_fat': 'body_fat'})
    raise ValueError(f'Could not identify the Body Fat columns. Columns found: {list(df.columns)}')

def safe_label(x) -> str:
    x = str(x).replace('.', 'p')
    x = re.sub('[^A-Za-z0-9_]+', '_', x)
    return x.strip('_')


## Prepare the data

In [ ]:
script_wall_start = time.perf_counter()
data = read_bodyfat(DATA_PATH)
data = data.apply(pd.to_numeric, errors='coerce').dropna().reset_index(drop=True)
y = data['body_fat'].to_numpy(dtype=float)
X = data[PREDICTORS].to_numpy(dtype=float)
n, p = X.shape
if p != 13:
    raise ValueError(f'Expected p=13 predictors, got p={p}.')
if STANDARDISE_X:
    X_mean = X.mean(axis=0, keepdims=True)
    X_sd = X.std(axis=0, ddof=1, keepdims=True)
    if np.any(X_sd == 0):
        raise ValueError('At least one predictor has zero standard deviation.')
    X = (X - X_mean) / X_sd
intercept = np.ones((n, 1), dtype=float)
X_full = np.column_stack([intercept, X])
XtX = X_full.T @ X_full
Xty = X_full.T @ y
yty = float(y @ y)
nu = n - 1
data.to_csv(OUT_DIR / 'bodyfat_data_used.csv', index=False)


## Prior and collapsed likelihood

In [ ]:
def beta_prior_from_expected_model_size(m0: float, p: int, b_pi: float=1.0):
    if not 0.0 < m0 < p:
        raise ValueError(f'Expected model size must be between 0 and p={p}, got {m0}.')
    prior_mean_pip = m0 / p
    a_pi = b_pi * prior_mean_pip / (1.0 - prior_mean_pip)
    initial_lambda = np.log(prior_mean_pip) - np.log1p(-prior_mean_pip)
    return {
        'a_pi': float(a_pi),
        'b_pi': float(b_pi),
        'prior_mean_pip': float(prior_mean_pip),
        'prior_expected_model_size': float(m0),
        'initial_rho': float(prior_mean_pip),
        'initial_lambda': float(initial_lambda),
    }

def log_marginal_likelihood_primal(mask: np.ndarray, tau2: float=TAU2) -> float:
    """
    Collapsed Gaussian marginal likelihood matching the VI active-set primal route.

    Model:
        y = beta0 * 1_n + X_gamma beta_gamma + eps
        eps ~ N(0, sigma^2 I)

    Priors:
        beta0 has flat prior;
        beta_gamma | sigma^2, gamma ~ N(0, sigma^2 * tau2 I);
        p(sigma^2) proportional to 1/sigma^2.
    """
    slope_idx = np.flatnonzero(mask) + 1
    idx = np.concatenate(([0], slope_idx))
    k_slope = len(slope_idx)
    q = k_slope + 1
    XtX_g = XtX[np.ix_(idx, idx)]
    Xty_g = Xty[idx]
    prior_diag = np.full(q, 1.0 / tau2, dtype=float)
    prior_diag[0] = 0.0
    M = XtX_g + np.diag(prior_diag)
    L = np.linalg.cholesky(M)
    logdet_M = 2.0 * np.sum(np.log(np.diag(L)))
    sol = np.linalg.solve(M, Xty_g)
    residual = yty - float(Xty_g @ sol)
    residual = max(residual, 1e-12)
    log_prior_prec = np.log(1.0 / tau2)
    return float(gammaln(0.5 * nu) - 0.5 * nu * np.log(2.0 * np.pi) + 0.5 * k_slope * log_prior_prec - 0.5 * logdet_M - 0.5 * nu * np.log(residual / 2.0))

def log_prior_gamma(mask: np.ndarray, a_pi: float, b_pi: float) -> float:
    k = int(mask.sum())
    return float(betaln(a_pi + k, b_pi + p - k) - betaln(a_pi, b_pi))

def model_string(mask: np.ndarray) -> str:
    selected = [PREDICTORS[j] for j in np.flatnonzero(mask)]
    return ' + '.join(selected) if selected else '(null)'


## Exact posterior calculation

In [ ]:
def exact_inference_core(m0: float):
    """
    Return the exact posterior quantities needed for the benchmark.

    Timed operations:
      1. enumerate all 2^p model indicators;
      2. evaluate collapsed log marginal likelihood for every model;
      3. evaluate integrated Beta-Bernoulli prior;
      4. normalise the exact posterior;
      5. calculate exact PIPs.

    No file I/O or report printing occurs here.
    """
    masks = np.zeros((2 ** p, p), dtype=bool)
    for m in range(2 ** p):
        masks[m, :] = [m >> j & 1 for j in range(p)]
    model_sizes = masks.sum(axis=1)
    log_ml_all = np.array([log_marginal_likelihood_primal(mask, TAU2) for mask in masks], dtype=float)
    prior = beta_prior_from_expected_model_size(m0, p=p, b_pi=B_PI_FIXED)
    a_pi = prior['a_pi']
    b_pi = prior['b_pi']
    log_pg = np.array([log_prior_gamma(mask, a_pi, b_pi) for mask in masks], dtype=float)
    log_post_unnorm = log_ml_all + log_pg
    logZ = float(logsumexp(log_post_unnorm))
    post_prob = np.exp(log_post_unnorm - logZ)
    exact_pips = masks.astype(float).T @ post_prob
    return {
        'masks': masks,
        'model_sizes': model_sizes,
        'log_ml_all': log_ml_all,
        'prior': prior,
        'a_pi': float(a_pi),
        'b_pi': float(b_pi),
        'log_pg': log_pg,
        'log_post_unnorm': log_post_unnorm,
        'logZ': logZ,
        'post_prob': post_prob,
        'exact_pips': exact_pips,
    }


## Benchmark inference

In [ ]:
if len(EXPECTED_MODEL_SIZE_GRID) != 1:
    raise ValueError('For the final comparable Body Fat runtime, EXPECTED_MODEL_SIZE_GRID should contain only m0=3.')
m0_final = float(EXPECTED_MODEL_SIZE_GRID[0])
runtime_seconds = []
result = None
print('=' * 100)
print('Benchmarking exact Body Fat enumeration')
print('=' * 100)
print(f'p = {p}; models per run = {2 ** p}; m0 = {m0_final}')
print(f'runtime repeats = {RUNTIME_REPEATS}')
print()
for r in range(RUNTIME_REPEATS):
    gc.collect()
    t0 = time.perf_counter()
    current = exact_inference_core(m0_final)
    elapsed = time.perf_counter() - t0
    runtime_seconds.append(float(elapsed))
    result = current
    print(f'Exact enumeration timing {r + 1}/{RUNTIME_REPEATS}: {elapsed:.6f} s')
runtime_seconds = np.asarray(runtime_seconds, dtype=float)
mean_runtime_seconds = float(runtime_seconds.mean())
sd_runtime_seconds = float(runtime_seconds.std(ddof=1)) if len(runtime_seconds) > 1 else np.nan
median_runtime_seconds = float(np.median(runtime_seconds))
min_runtime_seconds = float(runtime_seconds.min())
max_runtime_seconds = float(runtime_seconds.max())


## Build posterior summaries

In [ ]:
masks = result['masks']
model_sizes = result['model_sizes']
log_ml_all = result['log_ml_all']
prior = result['prior']
a_pi = result['a_pi']
b_pi = result['b_pi']
log_pg = result['log_pg']
log_post_unnorm = result['log_post_unnorm']
logZ = result['logZ']
post_prob = result['post_prob']
exact_pips = result['exact_pips']
label = safe_label(f'Esize_{m0_final}_Beta_{a_pi:.6g}_{b_pi:.6g}_tau2_{TAU2}_primal')
models = pd.DataFrame({
    'prior_label': label,
    'target_expected_model_size': m0_final,
    'a_pi': a_pi,
    'b_pi': b_pi,
    'tau2': TAU2,
    'model_index': np.arange(2 ** p),
    'model_size': model_sizes,
    'model': [model_string(mask) for mask in masks],
    'log_marginal_likelihood': log_ml_all,
    'log_prior_gamma': log_pg,
    'log_posterior_unnormalised': log_post_unnorm,
    'posterior_probability': post_prob,
})
for j, name in enumerate(PREDICTORS):
    models[f'gamma_{name}'] = masks[:, j].astype(int)
models = models.sort_values('posterior_probability', ascending=False).reset_index(drop=True)
models['cumulative_posterior_probability'] = models['posterior_probability'].cumsum()
pips = pd.DataFrame({
    'prior_label': label,
    'target_expected_model_size': m0_final,
    'a_pi': a_pi,
    'b_pi': b_pi,
    'tau2': TAU2,
    'variable': PREDICTORS,
    'exact_pip': exact_pips,
})
pips['selected_pip_gt_0p5'] = (pips['exact_pip'] > PIP_THRESHOLD).astype(int)
pips = pips.sort_values('exact_pip', ascending=False).reset_index(drop=True)
setting = {
    'prior_label': label,
    'target_expected_model_size': m0_final,
    **prior,
    'a_pi': a_pi,
    'b_pi': b_pi,
    'tau2': TAU2,
    'standardise_x': STANDARDISE_X,
    'likelihood_route': 'primal_intercept_raw_y',
    'posterior_expected_model_size': float(pips['exact_pip'].sum()),
    'selected_count_pip_gt_0p5': int((pips['exact_pip'] > PIP_THRESHOLD).sum()),
    'top_model_posterior_probability': float(models.loc[0, 'posterior_probability']),
    'top_10_cumulative_posterior_probability': float(models.head(10)['posterior_probability'].sum()),
    'log_evidence': logZ,
    'runtime_definition': 'exact inference only; excludes data loading/preprocessing and file I/O',
    'runtime_repeats': int(RUNTIME_REPEATS),
    'mean_runtime_seconds': mean_runtime_seconds,
    'sd_runtime_seconds': sd_runtime_seconds,
    'median_runtime_seconds': median_runtime_seconds,
    'min_runtime_seconds': min_runtime_seconds,
    'max_runtime_seconds': max_runtime_seconds,
}
settings_df = pd.DataFrame([setting])
selected_df = pips.loc[pips['exact_pip'] > PIP_THRESHOLD].copy()
top_models_df = models.head(TOP_MODELS_TO_SAVE).copy()
pips_wide = pips.pivot_table(index='variable', columns='prior_label', values='exact_pip').reset_index()


## Save posterior and runtime outputs

In [ ]:
models.to_csv(OUT_DIR / f'exact_model_posterior_{label}.csv', index=False)
settings_df.to_csv(OUT_DIR / 'prior_settings_by_expected_model_size.csv', index=False)
pips.to_csv(OUT_DIR / 'exact_pips_by_expected_size_long.csv', index=False)
pips_wide.to_csv(OUT_DIR / 'exact_pips_by_expected_size_wide.csv', index=False)
selected_df.to_csv(OUT_DIR / 'exact_selected_variables_by_expected_size.csv', index=False)
top_models_df.to_csv(OUT_DIR / 'exact_top_models_by_expected_size.csv', index=False)
runtime_repeats_df = pd.DataFrame({
    'method': 'Exact enumeration',
    'repeat': np.arange(1, RUNTIME_REPEATS + 1),
    'runtime_seconds': runtime_seconds,
})
runtime_repeats_df.to_csv(OUT_DIR / 'exact_enumeration_runtime_repeats.csv', index=False, float_format='%.17g')
exact_runtime_table = pd.DataFrame([
    {
        'method': 'Exact enumeration',
        'number_of_runs': int(RUNTIME_REPEATS),
        'mean_runtime_seconds': mean_runtime_seconds,
        'sd_runtime_seconds': sd_runtime_seconds,
        'median_runtime_seconds': median_runtime_seconds,
        'min_runtime_seconds': min_runtime_seconds,
        'max_runtime_seconds': max_runtime_seconds,
        'runtime_definition': 'exact inference only; excludes data loading/preprocessing and file I/O',
        'p': int(p),
        'number_of_models': int(2 ** p),
        'm0': float(m0_final),
        'a_pi': float(a_pi),
        'b_pi': float(b_pi),
        'tau2': float(TAU2),
    },
])
exact_runtime_table.to_csv(OUT_DIR / 'table_runtime_exact_enumeration.csv', index=False, float_format='%.17g')


## Optional combined runtime table

In [ ]:
ROOT = Path.cwd().resolve()
vi_runtime_candidates = [
    ROOT / 'bodyfat_FINAL_ORIGINAL_PSIS_DAMPED_IWMM_K5000_FULLCONST_V3' / 'table_runtime.csv',
    ROOT / 'bodyfat_FINAL_original_PSIS_damped_IWMM_K5000_FULLCONST_V3' / 'table_runtime.csv',
    ROOT / 'bodyfat_S1N300_PLUS_ELBO_EMPIRICAL_PSIS_ADAPTIVE_K' / 'table_runtime.csv',
    ROOT.parent / 'bodyfat_FINAL_ORIGINAL_PSIS_DAMPED_IWMM_K5000_FULLCONST_V3' / 'table_runtime.csv',
    ROOT.parent / 'bodyfat_FINAL_original_PSIS_damped_IWMM_K5000_FULLCONST_V3' / 'table_runtime.csv',
    ROOT.parent / 'bodyfat_S1N300_PLUS_ELBO_EMPIRICAL_PSIS_ADAPTIVE_K' / 'table_runtime.csv',
]
vi_runtime_file = next((path for path in vi_runtime_candidates if path.exists()), None)
combined_runtime_table = None
if vi_runtime_file is not None:
    vi_runtime = pd.read_csv(vi_runtime_file)
    if 'method' in vi_runtime.columns:
        vi_runtime = vi_runtime.loc[vi_runtime['method'].astype(str) != 'Exact enumeration'].copy()
    exact_row_for_vi = {c: np.nan for c in vi_runtime.columns}
    if 'method' in exact_row_for_vi:
        exact_row_for_vi['method'] = 'Exact enumeration'
    if 'number_of_runs' in exact_row_for_vi:
        exact_row_for_vi['number_of_runs'] = int(RUNTIME_REPEATS)
    if 'mean_runtime_seconds' in exact_row_for_vi:
        exact_row_for_vi['mean_runtime_seconds'] = mean_runtime_seconds
    if 'sd_runtime_seconds' in exact_row_for_vi:
        exact_row_for_vi['sd_runtime_seconds'] = sd_runtime_seconds
    if 'median_runtime_seconds' in exact_row_for_vi:
        exact_row_for_vi['median_runtime_seconds'] = median_runtime_seconds
    if 'min_runtime_seconds' in exact_row_for_vi:
        exact_row_for_vi['min_runtime_seconds'] = min_runtime_seconds
    if 'max_runtime_seconds' in exact_row_for_vi:
        exact_row_for_vi['max_runtime_seconds'] = max_runtime_seconds
    combined_runtime_table = pd.concat([vi_runtime, pd.DataFrame([exact_row_for_vi])], ignore_index=True)
    combined_runtime_table.to_csv(OUT_DIR / 'table_runtime_with_exact_enumeration.csv', index=False, float_format='%.17g')


## Runtime and results

In [ ]:
total_script_wall_seconds = float(time.perf_counter() - script_wall_start)
pd.DataFrame([
    {
        'exact_inference_mean_seconds': mean_runtime_seconds,
        'exact_inference_sd_seconds': sd_runtime_seconds,
        'total_script_wall_seconds': total_script_wall_seconds,
        'recommended_for_method_comparison': 'exact_inference_mean_seconds',
    },
]).to_csv(OUT_DIR / 'exact_runtime_audit.csv', index=False, float_format='%.17g')
print()
print('=' * 100)
print('Body Fat exact enumeration using primal/intercept route completed')
print('=' * 100)
print(f'Data path: {DATA_PATH}')
print(f'n = {n}, p = {p}, enumerated models = {2 ** p}')
print(f'm0 = {m0_final}, a_pi = {a_pi:.12g}, b_pi = {b_pi:.12g}')
print(f'tau2 = {TAU2}')
print(f'standardised X = {STANDARDISE_X}')
print('likelihood route = primal/intercept/raw-y')
print()
print('Exact-inference runtime (comparable with the VI training time)')
print(f'  mean   = {mean_runtime_seconds:.6f} s')
print(f'  SD     = {sd_runtime_seconds:.6f} s')
print(f'  median = {median_runtime_seconds:.6f} s')
print(f'  range  = [{min_runtime_seconds:.6f}, {max_runtime_seconds:.6f}] s')
print(f'  repeats = {RUNTIME_REPEATS}')
print()
print('Total script wall time (not used in the runtime comparison):')
print(f'  {total_script_wall_seconds:.6f} s')
print()
print('Exact-runtime table:')
print(exact_runtime_table[[
    'method',
    'number_of_runs',
    'mean_runtime_seconds',
    'sd_runtime_seconds',
    'median_runtime_seconds',
]].to_string(index=False))
if combined_runtime_table is not None:
    print()
    print('Found VI runtime table:')
    print(f'  {vi_runtime_file}')
    print()
    print('VI + Exact enumeration runtime table:')
    show_cols = [c for c in [
        'method',
        'number_of_runs',
        'mean_runtime_seconds',
        'sd_runtime_seconds',
        'median_runtime_seconds',
    ] if c in combined_runtime_table.columns]
    print(combined_runtime_table[show_cols].to_string(index=False))
    print()
    print('Saved combined table:')
    print(OUT_DIR / 'table_runtime_with_exact_enumeration.csv')
else:
    print()
    print('No existing table_runtime.csv was found in the known Body Fat result folders. The exact runtime was still saved separately.')
print()
print('Original exact PIPs:')
print(pips_wide.to_string(index=False))
print()
print('Selected variables using exact PIP > 0.5:')
if selected_df.empty:
    print('None')
else:
    print(selected_df[['variable', 'exact_pip']].to_string(index=False))
print()
print(f'Outputs saved to: {OUT_DIR.resolve()}')
